# PFE ML — Phase B: Hyperparameter Tuning Of The Top Two Libraries

Phase A established that HGB and CatBoost are the top two libraries at default hyperparameters (AP 0.155 / 0.148, top-0.1% precision 84.9% / 84.6%). LightGBM and XGBoost are dropped from Phase B because they were significantly behind in Phase A's operational metric (top-0.1% precision).

**Phase B question:** With a budget of 20 random configurations × 3-fold walk-forward CV, can either library improve over its default-setting AP?

## Methodological design

- **Search method:** `RandomizedSearchCV` with `n_iter=20`. RandomSearch is preferred over GridSearch for high-dimensional spaces because at this budget it explores the loss surface more efficiently.
- **CV strategy:** Walk-forward time splits, **not** random KFold. Three folds:
  - Fold 1: train 2017–2020, validate 2021
  - Fold 2: train 2017–2021, validate 2022
  - Fold 3: train 2017–2022, validate 2023
  - 2024 is reserved as the final held-out test set — never seen during tuning.
- **Scoring metric:** `average_precision`. This is the rare-event ranking metric and it's the one Phase A's headline number uses.
- **Data:** Same 2M-row deterministic hash sample as Runs 1, 2, 7, 8, 9, and Phase A. Apples-to-apples with everything that came before.
- **Class imbalance:** Each library's native knob, fixed at `balanced` semantics — not tuned. The tuning budget is reserved for the regularization-and-capacity dimensions.
- **Hardware:** HGB tunes on CPU (no GPU support). CatBoost tunes on GPU (`task_type='GPU'`). Final fit re-runs at full 2M with the same hardware as tuning.

## What this notebook produces

Two new entries appended to `model_run_comparison.csv` — `hgb_tuned` and `catboost_tuned` — alongside the 9 baseline runs + 4 Phase A runs, for **15 total comparable rows**. Plus two JSON files (`tuned_params_*.json`) with the best hyperparameters for reproducibility, and a comparison plot.

## What's needed on the branch

Before running, commit + push these on `data-extraction`:
1. `app/tools/train_continuity_model.py` exposes `--params-file PATH` (loads a JSON of classifier kwargs) and `--gpu` (CatBoost / XGBoost CUDA path).
2. `_build_model_pipeline()` accepts an `extra_params` override dict and a `gpu` flag.

## 1. Runtime

**Pick a GPU runtime** (T4 or A100 on Pro). CPU-only is fine too but CatBoost tuning will take roughly 4× longer.

- HGB tuning runs on CPU regardless (no GPU support in sklearn HGB). Expect ~70 min.
- CatBoost tuning runs on GPU when available. Expect ~20 min on T4, ~10 min on A100. On CPU it would be ~90 min.
- Total: ~90–100 min on T4, ~80 min on A100.

If you're tight on time, change `N_ITER` in section 4 from 20 to 15 to save 25% on each library.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/zribi1/pfein.git'
BRANCH = 'data-extraction'
REPO_DIR = '/content/pfein'
BACKEND_DIR = f'{REPO_DIR}/back_end'

DRIVE_ROOT = '/content/drive/MyDrive/PFE ML Data/pfe_data'
DATA_LAKE = f'{DRIVE_ROOT}/data-lake'
DUCKDB_TMP = '/content/pfein_duckdb_tmp'

START_YEAR = 2017
END_YEAR = 2024
TRAIN_MAX_ROWS = 2_000_000
TARGET = 'continuity_risk_12m_label'

N_ITER = 20            # configurations per library
N_SPLITS = 3           # walk-forward CV folds
TUNED_LIBRARIES = ['hgb', 'catboost']

FEATURES_GLOB = f'{DATA_LAKE}/features/company_year_features/**/*.parquet'
LABELS_GLOB = f'{DATA_LAKE}/features/risk_labels/**/*.parquet'

Path(DUCKDB_TMP).mkdir(parents=True, exist_ok=True)

print('BACKEND_DIR  =', BACKEND_DIR)
print('DRIVE_ROOT   =', DRIVE_ROOT)
print('TUNED LIBS   =', TUNED_LIBRARIES)
print('N_ITER       =', N_ITER, '/ library')
print('CV FOLDS     =', N_SPLITS, '(walk-forward)')

## 2. Pull Code And Install Dependencies

In [ ]:
import os

if not Path(REPO_DIR).exists():
    !git clone --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"

%cd $REPO_DIR
!git fetch origin
!git switch "$BRANCH" || git switch -c "$BRANCH" "origin/$BRANCH"
!git pull --ff-only origin "$BRANCH"
%cd $BACKEND_DIR

os.environ['DUCKDB_TEMP_DIRECTORY'] = DUCKDB_TMP
!pip install -q -r collabs/requirements-colab.txt

In [ ]:
train_script_text = (Path(BACKEND_DIR) / 'app' / 'tools' / 'train_continuity_model.py').read_text(encoding='utf-8')
checks = {
    '--params-file CLI flag': '--params-file' in train_script_text,
    '--gpu CLI flag': '--gpu' in train_script_text,
    'extra_params parameter': 'extra_params: dict' in train_script_text,
    'gpu parameter on pipeline builder': 'gpu: bool' in train_script_text,
}
print('Phase B readiness:')
for label, ok in checks.items():
    print(f'  {"OK " if ok else "FAIL"}  {label}')
if not all(checks.values()):
    raise SystemExit('Phase B plumbing is missing on the pulled branch. Commit + push and re-run.')

import torch
gpu_available = torch.cuda.is_available()
print(f'\nGPU available: {gpu_available}')
if gpu_available:
    print(f'GPU name: {torch.cuda.get_device_name(0)}')
else:
    print('Warning: GPU not detected. CatBoost will fall back to CPU and take ~4× longer.')

## 3. Load The Same 2M-Row Sample Used By Every Other Run

Replicates the exact query inside `train_continuity_model.py`: deterministic hash sample, time-ordered for the 2024 hold-out. Same `1,708,530` train rows and `291,470` test rows that Phase A used.

Reading the full 2M-row DataFrame in this notebook (instead of subprocessing the training script for each tuning fit) saves us from repeatedly re-reading parquet for every configuration of the search.

In [ ]:
import math, duckdb, pandas as pd, numpy as np
from app.tools.train_continuity_model import EXCLUDE_COLUMNS

filters = [f'l."{TARGET}" IS NOT NULL', f'f.prediction_year >= {START_YEAR}', f'f.prediction_year <= {END_YEAR}']
where_sql = ' AND '.join(filters)

con = duckdb.connect()
total_rows = con.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{FEATURES_GLOB}', union_by_name=true) f
    JOIN read_parquet('{LABELS_GLOB}', union_by_name=true) l USING (siren, prediction_year)
    WHERE {where_sql}
""").fetchone()[0]
print(f'Eligible rows: {total_rows:,}')

modulus = 1_000_000
threshold = max(1, min(modulus, math.ceil((TRAIN_MAX_ROWS / total_rows) * modulus * 1.15)))
row_hash = "hash(CAST(f.siren AS VARCHAR) || ':' || CAST(f.prediction_year AS VARCHAR))"
feature_cols = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{FEATURES_GLOB}', union_by_name=true)"
).df()['column_name'].tolist()
select_cols = [c for c in feature_cols if c not in EXCLUDE_COLUMNS]
select_sql = ', '.join(f'f."{c}"' for c in select_cols)

df = con.execute(f"""
    SELECT {select_sql}, l."{TARGET}"
    FROM read_parquet('{FEATURES_GLOB}', union_by_name=true) f
    JOIN read_parquet('{LABELS_GLOB}', union_by_name=true) l USING (siren, prediction_year)
    WHERE {where_sql} AND {row_hash} % {modulus} < {threshold}
    ORDER BY {row_hash}
    LIMIT {TRAIN_MAX_ROWS}
""").df()
con.close()

print(f'Loaded shape: {df.shape}')
print(f'Class balance: {df[TARGET].value_counts().to_dict()}')
print(f'Years: {sorted(df["prediction_year"].unique())}')

In [ ]:
y = df[TARGET].astype(int)
feature_columns = [c for c in df.columns if c not in EXCLUDE_COLUMNS and c != TARGET]
X = df[feature_columns].copy()
for col in X.columns:
    if pd.api.types.is_bool_dtype(X[col]):
        X[col] = X[col].astype(float)

numeric_columns = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
categorical_columns = [c for c in X.columns if c not in numeric_columns]
print(f'Numeric columns:     {len(numeric_columns)}')
print(f'Categorical columns: {len(categorical_columns)} ({categorical_columns})')

test_year = int(X['prediction_year'].max())
train_mask = X['prediction_year'] < test_year
X_train, X_test = X[train_mask].reset_index(drop=True), X[~train_mask].reset_index(drop=True)
y_train, y_test = y[train_mask].reset_index(drop=True), y[~train_mask].reset_index(drop=True)
print(f'\nTrain (years < {test_year}): {len(X_train):,} rows, {y_train.sum():,} positives')
print(f'Test  (year = {test_year}):   {len(X_test):,} rows, {y_test.sum():,} positives')

## 4. Walk-Forward CV Folds And Search Spaces

Year-by-year folds give honest temporal-generalization estimates. KFold on shuffled rows would underestimate the variance we care about (cross-year drift).

Search spaces are intentionally narrow around the Phase A defaults. We're not casting a wide net — we're refining around a known-good region.

In [ ]:
from scipy.stats import loguniform

def walk_forward_folds(X_train_df, year_column='prediction_year', n_splits=3):
    years = sorted(X_train_df[year_column].unique())
    folds = []
    for i in range(len(years) - n_splits, len(years)):
        if i < 2:
            continue
        train_years = years[:i]
        val_year = years[i]
        train_idx = np.where(X_train_df[year_column].isin(train_years))[0]
        val_idx = np.where(X_train_df[year_column] == val_year)[0]
        folds.append((train_idx, val_idx))
        print(f'  fold {len(folds)}: train years {train_years}, val year {val_year}'
              f' ({len(train_idx):,} train rows, {len(val_idx):,} val rows)')
    return folds

print(f'Walk-forward folds ({N_SPLITS}):')
cv_folds = walk_forward_folds(X_train, n_splits=N_SPLITS)

hgb_param_dist = {
    'classifier__learning_rate': loguniform(0.01, 0.2),
    'classifier__max_leaf_nodes': [15, 31, 63, 127],
    'classifier__min_samples_leaf': [20, 50, 100, 200, 500],
    'classifier__l2_regularization': [0.0, 0.5, 1.0, 2.0, 5.0, 10.0],
    'classifier__max_features': [0.5, 0.7, 1.0],
}

catboost_param_dist = {
    'classifier__learning_rate': loguniform(0.01, 0.2),
    'classifier__depth': [4, 6, 8, 10],
    'classifier__l2_leaf_reg': [1, 3, 5, 10, 30],
    'classifier__bagging_temperature': [0.0, 0.5, 1.0, 2.0],
}

print('\nHGB search space:')
for k, v in hgb_param_dist.items():
    print(f'  {k}: {v}')
print('\nCatBoost search space:')
for k, v in catboost_param_dist.items():
    print(f'  {k}: {v}')

## 5. Tune HGB (CPU)

Expect ~70 min. Each fold trains 20 candidate configurations, scored by average precision.

In [ ]:
import json, time, importlib
from sklearn.model_selection import RandomizedSearchCV
# Force-reload the training module so we pick up any source changes pulled in
# section 2 even when the kernel was started before the pull. Without this,
# Python's import cache keeps the stale ``_build_model_pipeline`` and sklearn
# ``clone()`` inside ``RandomizedSearchCV`` can trip identity checks on
# list-typed constructor params (e.g. CatBoost's ``class_weights`` /
# ``cat_features``) from the pre-pull module.
import app.tools.train_continuity_model as _tcm
importlib.reload(_tcm)
from app.tools.train_continuity_model import _build_model_pipeline

train_pos = int((y_train == 1).sum())
train_neg = int((y_train == 0).sum())

hgb_pipeline = _build_model_pipeline(
    family='hgb',
    categorical_columns=categorical_columns,
    train_positive_count=train_pos,
    train_negative_count=train_neg,
)

hgb_search = RandomizedSearchCV(
    hgb_pipeline,
    hgb_param_dist,
    n_iter=N_ITER,
    scoring='average_precision',
    cv=cv_folds,
    n_jobs=1,
    random_state=42,
    refit=False,
    return_train_score=False,
    verbose=2,
)

print(f'Starting HGB tuning: {N_ITER} configs x {len(cv_folds)} folds = {N_ITER * len(cv_folds)} fits')
start = time.time()
hgb_search.fit(X_train, y_train)
hgb_elapsed = time.time() - start
print(f'\nHGB tuning done in {hgb_elapsed/60:.1f} min')

hgb_best_params = {
    k.replace('classifier__', ''): (float(v) if isinstance(v, (np.floating,)) else v)
    for k, v in hgb_search.best_params_.items()
}
print(f'\nBest CV AP: {hgb_search.best_score_:.4f}')
print('Best params:')
for k, v in hgb_best_params.items():
    print(f'  {k}: {v}')

hgb_params_path = Path(DRIVE_ROOT) / 'ml-artifacts' / 'tuned_params_hgb.json'
hgb_params_path.parent.mkdir(parents=True, exist_ok=True)
hgb_params_path.write_text(json.dumps(hgb_best_params, indent=2), encoding='utf-8')
print(f'\nSaved: {hgb_params_path}')

## 6. Tune CatBoost (GPU)

Expect ~20 min on T4, ~10 min on A100. Each fold trains 20 candidate configurations on GPU.

In [ ]:
catboost_pipeline = _build_model_pipeline(
    family='catboost',
    categorical_columns=categorical_columns,
    train_positive_count=train_pos,
    train_negative_count=train_neg,
    gpu=gpu_available,
)

catboost_search = RandomizedSearchCV(
    catboost_pipeline,
    catboost_param_dist,
    n_iter=N_ITER,
    scoring='average_precision',
    cv=cv_folds,
    n_jobs=1,
    random_state=42,
    refit=False,
    return_train_score=False,
    verbose=2,
)

print(f'Starting CatBoost tuning ({"GPU" if gpu_available else "CPU"}): {N_ITER} configs x {len(cv_folds)} folds = {N_ITER * len(cv_folds)} fits')
start = time.time()
catboost_search.fit(X_train, y_train)
catboost_elapsed = time.time() - start
print(f'\nCatBoost tuning done in {catboost_elapsed/60:.1f} min')

catboost_best_params = {
    k.replace('classifier__', ''): (float(v) if isinstance(v, (np.floating,)) else v)
    for k, v in catboost_search.best_params_.items()
}
print(f'\nBest CV AP: {catboost_search.best_score_:.4f}')
print('Best params:')
for k, v in catboost_best_params.items():
    print(f'  {k}: {v}')

catboost_params_path = Path(DRIVE_ROOT) / 'ml-artifacts' / 'tuned_params_catboost.json'
catboost_params_path.write_text(json.dumps(catboost_best_params, indent=2), encoding='utf-8')
print(f'\nSaved: {catboost_params_path}')

## 7. Inspect The Search Surface

Before retraining the winners on full 2M, look at the CV scores to make sure the search wasn't dominated by one outlier configuration.

In [ ]:
def search_results_df(search, library):
    cv_results = pd.DataFrame(search.cv_results_)
    cv_results['library'] = library
    return cv_results[[
        'library', 'rank_test_score', 'mean_test_score', 'std_test_score',
        'params', 'mean_fit_time',
    ]].sort_values('rank_test_score').reset_index(drop=True)

hgb_results = search_results_df(hgb_search, 'hgb')
cb_results = search_results_df(catboost_search, 'catboost')

print('HGB — top 5 configurations:')
print(hgb_results.head(5).to_string(index=False))
print('\nCatBoost — top 5 configurations:')
print(cb_results.head(5).to_string(index=False))

combined_results = pd.concat([hgb_results, cb_results])
results_csv = Path(DRIVE_ROOT) / 'ml-artifacts' / 'tuning_search_results.csv'
combined_results.to_csv(results_csv, index=False)
print(f'\nSaved full search results to: {results_csv}')

In [ ]:
import matplotlib.pyplot as plt

tuning_dir = Path(DRIVE_ROOT) / 'ml-artifacts' / 'tuning_phase_b'
tuning_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (lib, results) in zip(axes, [('hgb', hgb_results), ('catboost', cb_results)]):
    scores = results['mean_test_score'].values
    stds = results['std_test_score'].values
    x = np.arange(len(scores))
    ax.errorbar(x, scores, yerr=stds, fmt='o', color='#0f766e', alpha=0.7)
    ax.axhline(scores.max(), color='#b91c1c', linestyle='--', label=f'best: {scores.max():.4f}')
    ax.set_title(f'{lib} — CV average precision per configuration')
    ax.set_xlabel('Configuration rank (best first)')
    ax.set_ylabel('CV mean AP (3-fold walk-forward)')
    ax.grid(True, alpha=0.3)
    ax.legend()
fig.tight_layout()
fig.savefig(tuning_dir / 'tuning_score_distribution.png', dpi=160)
plt.show()

## 8. Retrain Winners On Full 2M With Tuned Params

Invokes `train_continuity_model.py --params-file ... --model-family ...` for each library so the tuned runs land in the standard artifact pipeline (`runs/*/` folder, `model_run_comparison.csv` row, `feature_importances.csv`, plots, etc.) alongside all prior runs.

Each retrain is a **single fit** on the full 2M sample with the tuned hyperparameters — no CV at this stage. The 2024 hold-out from sections 3–4 is what these models are scored against.

In [ ]:
import shlex, subprocess, sys

retrain_results = {}
for family, params_path in [('hgb', hgb_params_path), ('catboost', catboost_params_path)]:
    print('=' * 70)
    print(f'Retraining {family} with tuned hyperparameters')
    print('=' * 70)
    cmd = [
        sys.executable, '-u',
        '-m', 'app.tools.train_continuity_model',
        '--data-lake-dir', f'{DRIVE_ROOT}/data-lake',
        '--artifacts-dir', f'{DRIVE_ROOT}/ml-artifacts',
        '--target', TARGET,
        '--train-start-year', str(START_YEAR),
        '--train-end-year', str(END_YEAR),
        '--max-rows', str(TRAIN_MAX_ROWS),
        '--min-rows', '1000',
        '--model-family', family,
        '--params-file', str(params_path),
    ]
    if family == 'catboost' and gpu_available:
        cmd.append('--gpu')
    print(' '.join(shlex.quote(p) for p in cmd))
    start = time.time()
    subprocess.run(cmd, check=True)
    elapsed = time.time() - start
    metadata = json.loads((Path(DRIVE_ROOT) / 'ml-artifacts' / 'model_metadata.json').read_text(encoding='utf-8'))
    retrain_results[family] = {
        'run_name': metadata['run_name'],
        'run_dir': metadata['run_artifacts_dir'],
        'elapsed_seconds': elapsed,
        'metrics': metadata['metrics'],
    }
    m = metadata['metrics']
    print(
        f"\nDone ({elapsed:.0f}s). "
        f"AUC={m['roc_auc']:.4f}  "
        f"AP={m['average_precision']:.4f}  "
        f"F1@0.5={m['f1_at_0_5']:.4f}\n"
    )

print('Both tuned models retrained.')

## 9. Tuned vs Default — Side By Side

In [ ]:
comparison_csv = Path(DRIVE_ROOT) / 'ml-artifacts' / 'model_run_comparison.csv'
all_runs = pd.read_csv(comparison_csv)

phase_a_runs = all_runs[all_runs['run_name'].str.contains('hgb_time-test-2024_cap-2m', na=False) |
                       all_runs['run_name'].str.contains('catboost_time-test-2024_cap-2m', na=False)].copy()
phase_a_runs = phase_a_runs.sort_values('trained_at').reset_index(drop=True)

summary_cols = [
    'run_name', 'model_family',
    'average_precision', 'roc_auc', 'precision_at_0_5', 'recall_at_0_5', 'f1_at_0_5',
]
summary_cols = [c for c in summary_cols if c in phase_a_runs.columns]
print('Phase A defaults + Phase B tuned (chronological):')
print(phase_a_runs[summary_cols].to_string(index=False))

def metric_delta(family, metric):
    rows = phase_a_runs[phase_a_runs['model_family'] == family]
    if len(rows) < 2:
        return None
    default = rows[metric].iloc[0]
    tuned = rows[metric].iloc[-1]
    return default, tuned, tuned - default

deltas = []
for family in TUNED_LIBRARIES:
    for metric in ['roc_auc', 'average_precision', 'f1_at_0_5']:
        result = metric_delta(family, metric)
        if result is None:
            continue
        default, tuned, delta = result
        deltas.append({
            'library': family, 'metric': metric,
            'default': default, 'tuned': tuned, 'delta': delta,
            'delta_pct': 100 * delta / default if default else None,
        })
deltas_df = pd.DataFrame(deltas)
print('\nTuned vs default deltas:')
print(deltas_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
metrics_to_plot = ['roc_auc', 'average_precision', 'f1_at_0_5']

for ax, metric in zip(axes, metrics_to_plot):
    width = 0.35
    x = np.arange(len(TUNED_LIBRARIES))
    defaults_vals = []
    tuned_vals = []
    for family in TUNED_LIBRARIES:
        rows = phase_a_runs[phase_a_runs['model_family'] == family].sort_values('trained_at')
        defaults_vals.append(rows[metric].iloc[0] if len(rows) > 0 else 0)
        tuned_vals.append(rows[metric].iloc[-1] if len(rows) > 1 else 0)
    ax.bar(x - width/2, defaults_vals, width, label='default', color='#94a3b8')
    ax.bar(x + width/2, tuned_vals, width, label='tuned', color='#0f766e')
    ax.set_xticks(x, TUNED_LIBRARIES)
    ax.set_title(metric)
    ax.grid(True, axis='y', alpha=0.3)
    ax.legend()
    for xi, (d, t) in enumerate(zip(defaults_vals, tuned_vals)):
        ax.text(xi - width/2, d, f'{d:.4f}', ha='center', va='bottom', fontsize=8)
        ax.text(xi + width/2, t, f'{t:.4f}', ha='center', va='bottom', fontsize=8)
fig.suptitle('Phase B: tuned vs default per library')
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(tuning_dir / 'tuned_vs_default.png', dpi=160)
plt.show()

## 10. Display The Full 15-Run Comparison

In [ ]:
keep = [
    'run_name', 'model_family', 'rows', 'feature_count',
    'average_precision', 'roc_auc', 'precision_at_0_5', 'recall_at_0_5', 'f1_at_0_5',
]
keep = [c for c in keep if c in all_runs.columns]
print('All runs (chronological):')
print(all_runs[keep].to_string(index=False))
print(f'\nTotal runs: {len(all_runs)}')
print('Last two rows are the Phase B tuned models.')

## 11. What To Send / What To Decide

### Attach to the thesis
- `ml-artifacts/tuned_params_hgb.json`, `tuned_params_catboost.json` — final hyperparameters for reproducibility.
- `ml-artifacts/tuning_search_results.csv` — full CV results across all 40 configurations (20 per library).
- `ml-artifacts/tuning_phase_b/tuning_score_distribution.png` — shows how flat or peaked the search surface was per library. A flat surface = defaults were already good; a peaked surface = tuning had a real effect.
- `ml-artifacts/tuning_phase_b/tuned_vs_default.png` — the headline before/after chart.
- The two newest `runs/*/` folders — each contains `run_summary.md`, `feature_importances.csv`, and the standard plots for the tuned models.

### Decide
Compare the **tuned** Phase B numbers with the **default** Phase A numbers (Run 9 = hgb default, Run 13 = catboost default):
- If tuned AP improves by ≥1 pp over default for a library → tuning was worth the budget; report tuned numbers as final.
- If tuned AP improves by <0.5 pp → defaults were already near-optimal; report this as a methodological finding ("we verified the default-setting result was robust to hyperparameter perturbation").
- If tuned AP *regresses* — the search overfit the CV (unlikely with 20 configs but possible). Report and investigate.

### Thesis framing for Phase B
*"Hyperparameter tuning with 20 random configurations and 3-fold walk-forward time-aware CV yielded a tuned HGB model with AP X.XX (vs Y.YY default, ΔZ pp) and a tuned CatBoost with AP X.XX (ΔZ pp). The walk-forward CV strategy is appropriate for the time-shifted prediction problem, where standard KFold would underestimate variance from cross-year drift. Both libraries were tuned with comparable budgets to ensure fair comparison."*

After Phase B, the natural next steps are **Phase C** (temporal stability — retrain the winner with 2023 as test instead of 2024, see if AUC stays in the same range) and **Phase D** (SHAP-based interpretability for the winner).